# Задание 6. Разбиение и признаки

Объяснение TF-IDF и N-грамм: `docs/06_features_explained.md`. Код признаков: `ml/features.py`.

In [1]:
# Настройка путей. Работает локально и в Google Colab.
import sys
import zipfile
from pathlib import Path

if "google.colab" in sys.modules and not Path("ml").exists():
    # В Colab загрузите архив проекта sms_guard_colab.zip (см. README).
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")

ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
FIGURES = ROOT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Корень проекта:", ROOT.name)

Корень проекта: AIproject


In [2]:
import numpy as np
import pandas as pd

from ml.data.split_sms import load_datasets, make_split, save_split
from ml.features import HAND_FEATURE_NAMES, build_feature_union

clean = pd.read_csv(ROOT / "data" / "processed" / "sms_clean.csv", encoding="utf-8")
save_split(make_split(clean))
data = load_datasets()
pd.DataFrame({
    name: {"Записей": len(part), "Доля выборки": len(part) / len(clean), "Доля spam": part["label"].mean()}
    for name, part in data.items()
}).T.round(4)

,Записей,Доля выборки,Доля spam
train,3594.0,0.6999,0.1205
val,770.0,0.1500,0.1195
test,771.0,0.1501,0.1206


## Построение матриц

`fit` вызывается **только на train**; val и test проходят через `transform`.

In [3]:
union = build_feature_union()
X_train = union.fit_transform(data["train"]["text_raw"])
X_val = union.transform(data["val"]["text_raw"])
X_test = union.transform(data["test"]["text_raw"])

word = dict(union.transformer_list)["word"]
char = dict(union.transformer_list)["char"]
print("Слова (1–2 граммы):", len(word.vocabulary_))
print("Символы (3–5 граммы):", len(char.vocabulary_))
print("Ручные признаки:", HAND_FEATURE_NAMES)
for name, matrix, part in [("X_train", X_train, "train"), ("X_val", X_val, "val"), ("X_test", X_test, "test")]:
    assert matrix.shape[0] == len(data[part])
    print(name, matrix.shape)
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

Слова (1–2 граммы): 9116
Символы (3–5 граммы): 23791
Ручные признаки: ['length', 'upper_ratio', 'exclamation_count', 'has_url', 'has_money', 'has_phone', 'rule_group_count']
X_train (3594, 32914)
X_val (770, 32914)
X_test (771, 32914)


## Топ-15 признаков с наибольшим средним TF-IDF по классам (обучающая выборка)

In [4]:
X_word = word.transform(data["train"]["text_raw"])
names = np.array(word.get_feature_names_out())
y = data["train"]["label"].to_numpy()
table = {}
for label, title in [(0, "ham"), (1, "spam")]:
    means = np.asarray(X_word[y == label].mean(axis=0)).ravel()
    order = means.argsort()[::-1][:15]
    table[f"{title}: признак"] = names[order]
    table[f"{title}: средний TF-IDF"] = means[order].round(4)
pd.DataFrame(table)

,ham: признак,ham: средний TF-IDF,spam: признак,spam: средний TF-IDF
0,you,0.0309,<num>,0.1090
1,?,0.0296,<money>,0.0599
2,to,0.0241,<phone>,0.0527
3,the,0.0203,to,0.0463
4,<num>,0.0186,!,0.0429
5,in,0.0182,call,0.0415
6,me,0.0175,<num> <num>,0.0357
7,and,0.0170,your,0.0330
8,it,0.0162,to <num>,0.0313
9,my,0.0160,call <phone>,0.0306
